# Evolution log — py-dynwrap

> Per-iteration narrative of the v0.1 port.

In [1]:
history = []
def _record(i, title, parity, status, narrative):
    history.append({'iter': i, 'title': title, 'parity': parity, 'status': status, 'narrative': narrative})

## Iteration 0 — Baseline core data wrapper

Ported `wrap_data`, `add_trajectory`, `add_linear_trajectory`, `add_branch_trajectory`, `add_cluster_graph` + helpers. The `Trajectory` dataclass mirrors R `dynwrap`'s list-based structure with explicit Optional fields. First parity attempt — pre-geodesic — just confirmed cell_ids and milestone_ids round-trip correctly between R and Py.

In [2]:
_record(0, 'Baseline data wrapper', 1.0, 'accepted', 'cell_ids/milestone_ids round-trip')

## Iteration 1 — convert_* functions (percentages ↔ progressions)

Implemented bidirectional conversion. For percentages → progressions, picks the edge whose endpoints account for the most weight (matching R's `convert_milestone_percentages_to_progressions`). For progressions → percentages, splits each cell's percentage between edge endpoints and renormalises per cell.

In [3]:
_record(1, 'Bidirectional percentage/progression converters', 1.0, 'accepted', 'milestone_percentages round-trips with R')

## Iteration 2 — calculate_geodesic_distances (the hard one)

Ported the tent-aware Dijkstra:
1. Auto-generate extra divergence_regions from edges not in any existing divergence
2. Build per-tent cell distance matrix via Manhattan in tent-percentage simplex
3. Combine milestone edges + cell-tent stubs + intra-tent cell edges → weighted graph
4. Dijkstra from each waypoint

First attempt failed with `KeyError: 'Index([1.0, 0.0]) not in columns'` — the empty divergence_regions DataFrame had object-typed columns, and concatenating bool extra rows produced mixed dtypes that pandas couldn't use as boolean indexer. Fixed by `dr['is_start'] = dr['is_start'].astype(bool)` after the concat.

In [4]:
_record(2, 'Bool dtype fix in calculate_geodesic_distances', 1.0, 'rejected→accepted', 'KeyError on linear trajectory; fixed via astype(bool) post-concat')

## Iteration 3 — R parity confirmed byte-equivalent

Ran R `dynwrap::calculate_geodesic_distances` and Py `pydynwrap.calculate_geodesic_distances` on the same canonical 3-branch fixture. **max|Δ| = 0.0** across 30×30 matrix. The networkx Dijkstra + tent-construction reproduces R igraph's output exactly.

In [5]:
_record(3, 'Byte-equivalent vs R on 30×30 fixture', 1.0, 'accepted', 'max|Δ|=0.0')
print(history)

[{'iter': 0, 'title': 'Baseline data wrapper', 'parity': 1.0, 'status': 'accepted', 'narrative': 'cell_ids/milestone_ids round-trip'}, {'iter': 1, 'title': 'Bidirectional percentage/progression converters', 'parity': 1.0, 'status': 'accepted', 'narrative': 'milestone_percentages round-trips with R'}, {'iter': 2, 'title': 'Bool dtype fix in calculate_geodesic_distances', 'parity': 1.0, 'status': 'rejected→accepted', 'narrative': 'KeyError on linear trajectory; fixed via astype(bool) post-concat'}, {'iter': 3, 'title': 'Byte-equivalent vs R on 30×30 fixture', 'parity': 1.0, 'status': 'accepted', 'narrative': 'max|Δ|=0.0'}]
